[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module2/06-debugging.ipynb)

# Debugging & Profiling
**Module 2 — Intermediate Python | Estimated time: 25 minutes**

## Learning Objectives
- Use **`pdb`** / `breakpoint()` to step through code interactively
- Apply `pdb` commands: `n`, `s`, `c`, `l`, `p`, `q`, `bt`
- Perform **post-mortem debugging** after an exception
- Use the **`logging`** module with levels, handlers, formatters, and structured logging
- Profile CPU usage with **`cProfile`** and `pstats`
- Measure line-level hotspots with **`line_profiler`**
- Follow the debugging progression: print → logging → breakpoints → profiling

In [ ]:
!pip install line_profiler --quiet
import logging
import cProfile
import pstats
import io
import sys
print('Setup complete.')

## 1. Print Debugging — The Starting Point

Everyone starts here. It works, but quickly becomes noisy and hard to remove.  
We will deliberately progress to better tools.

In [ ]:
def parse_config(text: str) -> dict:
    """Parse a simple KEY=VALUE config string."""
    result = {}
    for line_num, line in enumerate(text.strip().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith('#'):    # skip blank lines and comments
            continue

        # --- print debugging ---
        print(f'[DEBUG line {line_num}] Processing: {line!r}')

        if '=' not in line:
            print(f'[WARN  line {line_num}] Skipping malformed line: {line!r}')
            continue

        key, _, value = line.partition('=')
        result[key.strip()] = value.strip()

    print(f'[DEBUG] Final result: {result}')
    return result


config_text = """
# App settings
HOST = localhost
PORT = 8080
DEBUG = true
badline
"""

cfg = parse_config(config_text)
print('\nParsed config:', cfg)

## 2. The `logging` Module — Better Than Print

`logging` lets you set severity levels, route messages to multiple destinations, and toggle verbosity without editing the code.

In [ ]:
import logging

# Basic configuration — only messages at WARNING or above appear by default
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)-8s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
    force=True,          # override any existing configuration in the notebook
)

logger = logging.getLogger('pypath.config')


def parse_config_logged(text: str) -> dict:
    """Config parser using logging instead of print."""
    result = {}
    for line_num, line in enumerate(text.strip().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith('#'):
            logger.debug('Line %d: skipped (blank/comment)', line_num)
            continue
        if '=' not in line:
            logger.warning('Line %d: malformed, no "=" found: %r', line_num, line)
            continue
        key, _, value = line.partition('=')
        result[key.strip()] = value.strip()
        logger.debug('Line %d: parsed %r = %r', line_num, key.strip(), value.strip())

    logger.info('Parsed %d key(s) from config', len(result))
    return result


config_text = """
# App settings
HOST = localhost
PORT = 8080
badline
"""

cfg = parse_config_logged(config_text)
print('Result:', cfg)

## 3. Handlers and Formatters — Routing Logs

Send logs to a file, the console, or both — with different formats for each.

In [ ]:
import logging
import sys

# Create a dedicated logger (don't use root logger in libraries)
app_logger = logging.getLogger('myapp')
app_logger.setLevel(logging.DEBUG)
app_logger.handlers.clear()          # remove any existing handlers (notebook safety)

# Console handler — INFO and above, concise format
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))

# File handler — DEBUG and above, detailed format
file_handler = logging.FileHandler('/tmp/app.log', mode='w')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(
    logging.Formatter('%(asctime)s %(name)s %(levelname)s %(message)s')
)

app_logger.addHandler(console_handler)
app_logger.addHandler(file_handler)
app_logger.propagate = False          # don't pass to root logger

app_logger.debug('This goes to file only (below INFO threshold for console)')
app_logger.info('Server started on port 8080')
app_logger.warning('Config file not found, using defaults')
app_logger.error('Failed to connect to database: timeout after 30s')

print('\n--- Contents of /tmp/app.log ---')
with open('/tmp/app.log') as f:
    print(f.read())

## 4. Structured Logging

For production systems it is common to log JSON so that log aggregators (Splunk, Datadog, etc.) can parse fields automatically.

In [ ]:
import logging
import json
import time

class JSONFormatter(logging.Formatter):
    """Emit log records as single-line JSON objects."""

    def format(self, record: logging.LogRecord) -> str:
        payload = {
            'ts':      self.formatTime(record, '%Y-%m-%dT%H:%M:%S'),
            'level':   record.levelname,
            'logger':  record.name,
            'message': record.getMessage(),
        }
        # Merge in any extra fields passed via the `extra` kwarg
        for key, value in record.__dict__.items():
            if key not in logging.LogRecord.__dict__ and not key.startswith('_'):
                payload[key] = value
        if record.exc_info:
            payload['exception'] = self.formatException(record.exc_info)
        return json.dumps(payload)


json_logger = logging.getLogger('json_demo')
json_logger.setLevel(logging.DEBUG)
json_logger.handlers.clear()
json_logger.propagate = False
json_handler = logging.StreamHandler()
json_handler.setFormatter(JSONFormatter())
json_logger.addHandler(json_handler)

json_logger.info('User logged in', extra={'user_id': 42, 'ip': '192.168.1.1'})
json_logger.warning('Rate limit approached', extra={'endpoint': '/api/search', 'requests': 95, 'limit': 100})
try:
    1 / 0
except ZeroDivisionError:
    json_logger.error('Unexpected error in handler', exc_info=True)

## 5. `pdb` — The Python Debugger

In a notebook, `pdb` works best via `pdb.run()` or `pdb.post_mortem()`.  
The interactive `breakpoint()` call is more useful in scripts — we will demonstrate the commands here.

In [ ]:
import pdb
import traceback

# Demonstrate pdb commands via a scripted session
# In a real script you would insert `breakpoint()` and run from the terminal.
# Key commands:
#   n  (next)      — execute current line, stay at same level
#   s  (step)      — step INTO function calls
#   c  (continue)  — run until next breakpoint or end
#   l  (list)      — show surrounding source code
#   p <expr>       — print the value of an expression
#   pp <expr>      — pretty-print
#   bt             — print the call stack (backtrace)
#   u / d          — move up/down the call stack
#   q  (quit)      — exit the debugger

# Post-mortem debugging — inspect state after an unhandled exception
def buggy_function(data: list) -> float:
    total = 0
    for item in data:
        total += item['value']   # KeyError if 'value' key is missing
    return total / len(data)


test_data = [
    {'value': 10},
    {'value': 20},
    {'val': 30},    # BUG: wrong key
]

try:
    result = buggy_function(test_data)
except Exception as exc:
    print(f'Exception caught: {type(exc).__name__}: {exc}')
    print('\nTraceback:')
    traceback.print_exc()
    print('\n[In a script, you would now call pdb.post_mortem() to inspect the frame.]')
    print('[In Colab, use pdb.post_mortem(sys.exc_info()[2]) in a terminal.]')
    # pdb.post_mortem()  # <- uncomment when running in a terminal / script

## 6. Profiling with `cProfile`

`cProfile` measures how many times each function was called and how long was spent in each.

In [ ]:
import cProfile
import pstats
import io

# A deliberately inefficient function to profile
def slow_primes(limit: int) -> list[int]:
    """Return all primes up to `limit` using a naive trial-division sieve."""
    primes = []
    for n in range(2, limit + 1):
        is_prime = all(n % i != 0 for i in range(2, int(n**0.5) + 1))
        if is_prime:
            primes.append(n)
    return primes


# Profile using cProfile
profiler = cProfile.Profile()
profiler.enable()
result = slow_primes(5000)
profiler.disable()

print(f'Found {len(result)} primes up to 5000')

# Analyse results with pstats
stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats('cumulative')       # sort by cumulative time
stats.print_stats(10)               # top 10 functions
print(stream.getvalue())

## 7. `line_profiler` — Line-Level Hotspots

When `cProfile` tells you *which function* is slow, `line_profiler` tells you *which line*.

In [ ]:
%%writefile /tmp/profile_target.py
import math

def process_records(records):
    """Process a list of numerical records."""
    results = []
    for r in records:               # line A
        x = math.sqrt(r)            # line B
        y = x ** 2 + 2 * x + 1     # line C
        z = math.log(y + 1)         # line D
        results.append(z)           # line E
    return results

if __name__ == '__main__':
    data = list(range(1, 100_001))
    out = process_records(data)
    print(f'Processed {len(out)} records, last value = {out[-1]:.4f}')

In [ ]:
# Run kernprof (line_profiler CLI) on the file
!kernprof -l -v /tmp/profile_target.py 2>&1

In [ ]:
# Alternative: use line_profiler inside the notebook with the magic
%load_ext line_profiler

import sys
sys.path.insert(0, '/tmp')
from profile_target import process_records

data = list(range(1, 50_001))
%lprun -f process_records process_records(data)

## 8. Debugging Strategy Summary

A practical progression for diagnosing a performance or correctness issue:

In [ ]:
# Strategy walkthrough: diagnose why a report generation is slow

import time
import logging
import cProfile, pstats, io

logging.basicConfig(level=logging.DEBUG, format='[%(levelname)s] %(message)s', force=True)
log = logging.getLogger('report')


def fetch_rows(n: int) -> list[dict]:
    log.debug('Fetching %d rows from database', n)
    time.sleep(0.01)  # simulate I/O
    return [{'id': i, 'value': i * 1.5} for i in range(n)]

def compute_totals(rows: list[dict]) -> float:
    log.debug('Computing totals for %d rows', len(rows))
    return sum(r['value'] for r in rows)   # generator — memory efficient

def format_report(total: float, count: int) -> str:
    log.debug('Formatting report')
    return f'Report: {count} records, total = {total:,.2f}'


def generate_report(n: int = 500) -> str:
    """Full pipeline with logging at each step."""
    log.info('Starting report generation for n=%d', n)
    t0 = time.perf_counter()

    rows = fetch_rows(n)
    log.info('Fetch took %.3fs', time.perf_counter() - t0)

    total = compute_totals(rows)
    report = format_report(total, len(rows))

    log.info('Report complete in %.3fs', time.perf_counter() - t0)
    return report


# Step 1: quick timing
print(generate_report(200))

# Step 2: cProfile to find the bottleneck
prof = cProfile.Profile()
prof.enable()
generate_report(200)
prof.disable()
buf = io.StringIO()
pstats.Stats(prof, stream=buf).sort_stats('cumulative').print_stats(5)
print(buf.getvalue())

# Step 3: if the profiler shows fetch_rows is slow → optimise DB query or add caching
# Step 4: if compute_totals is slow → use numpy or a compiled extension
print('Debugging strategy: logging → cProfile → line_profiler → optimise')

## Practice Exercises

**Exercise 1 — Add Logging to Existing Code**  
Take the `parse_config` function from Section 1 and replace all `print` statements with appropriate `logging` calls (DEBUG for trace info, WARNING for malformed lines, INFO for the summary). Set up a `FileHandler` that writes to `/tmp/config_parser.log` and a `StreamHandler` that shows only WARNING and above on the console.

**Exercise 2 — Profile and Optimise**  
Write a function `count_words(text: str) -> dict[str, int]` that counts word frequencies. First implement it naively (split → loop → dict), profile it with `cProfile` on a 50,000-word text, then rewrite it using `collections.Counter`. Compare the profile outputs.

**Exercise 3 — Post-Mortem Debugging Script**  
Write a Python *script* (not notebook) to `/tmp/debug_me.py` that contains a subtle bug (e.g., an off-by-one error in a list index). Add a `try/except` that calls `pdb.post_mortem()` in the `except` block. Run it with `!python /tmp/debug_me.py` and describe the `pdb` session commands you would use to diagnose the bug.